## Prepare pipeline

This pipeline is responsible for preparing, cleaning and parsing the data.

It reads the PARQUET files individually from BRONZE layer and append to the SILVER layer, specifically to the 'garbage_collection_silver_lakehouse' delta lakehouse.

This pipeline is supposed to run right after each EXTRACT pipeline, operating each file individually.

In [0]:
import os
from pyspark.sql.types import StructType, StructField, StringType
from typing import Optional
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

bronze_directory = "/Volumes/workspace/default/garabage_collection_volume/bronze/"
silver_directory = "/Volumes/workspace/default/garabage_collection_volume/silver/"
garbage_collection_silver_lakehouse_directory = os.path.join(silver_directory, "garbage_collection_silver_lakehouse.delta")    

In [0]:
# Month utils
month_map = {
    "janeiro": "01", "jan": "01",
    "fevereiro": "02", "fev": "02",
    "março": "03", "marco": "03", "mar": "03",  # handle 'ç' and 'c'
    "abril": "04", "abr": "04",
    "maio": "05", "mai": "05",
    "junho": "06", "jun": "06",
    "julho": "07", "jul": "07",
    "agosto": "08", "ago": "08",
    "setembro": "09", "set": "09",
    "outubro": "10", "out": "10",
    "novembro": "11", "nov": "11",
    "dezembro": "12", "dez": "12",
}
mapping_expr = F.create_map(*[x for kv in month_map.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])
original_month_col_normalized = F.lower(F.trim(F.regexp_replace("MES_DESCRITIVO", r"ç", "c")))

# ABNT Decilmal to Float Utils
def abnt_decimal_to_float(df: DataFrame, col: str) -> DataFrame:
    
    df = df.withColumn(col, F.trim(F.col(col)))
    df = df.withColumn(col, F.when(F.col(col) == "", "0").otherwise(F.col(col)))
    df = df.fillna("0", subset=[col])
    df = df.withColumn(col, F.regexp_replace(F.col(col), r"\.", ""))
    df = df.withColumn(col, F.regexp_replace(F.col(col), ",", ".").cast("double"))  
    df = df.withColumn(col, F.round(F.col(col), 2))
    return df


In [0]:
def extract(file_name: str) -> DataFrame:    
    file_to_prepare = os.path.join(bronze_directory, f"{file_name}.parquet")
    df_file_to_prepare = spark.read.parquet(file_to_prepare)
    return df_file_to_prepare

def transform(df: DataFrame) -> DataFrame:
    df = df.withColumn("ANO", df.ANO.cast("int"))
    df = df.withColumn("VIAGENS", df.VIAGENS.cast("int"))
    df = df.withColumn("DIAS", df.DIAS.cast("int"))
    df = df.withColumn("COLETORES", df.COLETORES.cast("int"))

    df = abnt_decimal_to_float(df, "MASSA")
    df = abnt_decimal_to_float(df, "PERCURSO")
    df = abnt_decimal_to_float(df, "HORAS")
    
    df = df.withColumn("MES_DESCRITIVO", df.MES)
    df = df.withColumn("MES", F.element_at(mapping_expr, original_month_col_normalized))
    return df

def load(df: DataFrame):    
    df.write.format("delta").mode("append").save(garbage_collection_silver_lakehouse_directory)


In [0]:
file_name = dbutils.jobs.taskValues.get(key="file_name", taskKey="Extraction")

df = extract(file_name)
df = transform(df)
load(df)
